<a href="https://colab.research.google.com/github/msrehman786/IBM-AI-Certification/blob/main/Chatbot_Building_a_Modern_LLM_Chatbot_with_Chat_Templates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Installing requirements

In [ ]:
!pip3 install virtualenv
!virtualenv my_env # create a virtual environment my_env
!source my_env/bin/activate # activate my_env

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 23.3 MB/s eta 0:00:00
created virtual environment CPython3.13.15.final.0-64-x86_64 in 524ms
  creator CPython3Posix(dest=/content/my_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator


In [ ]:
!pip install transformers==4.41.2 torch==2.12 accelerate==0.30.1 numpy==1.26.4

  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 72.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 24.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 892.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━

Step 1: Import required libraries
At first, start by creating a new file named chatbot_llm.py

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

Step 2: Choose a modern LLM

In [2]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

Step 3: Load model and tokenizer

pad_token: In transformer models, inputs in a batch must often be the same length. Shorter sequences are padded with a special token called the padding token (pad_token). This tells the model which parts of the input are real words and which are filler.
device_map: Controls where the model runs (e.g., CPU or GPU) and ensures it is correctly loaded on the available device.
torch_dtype: Sets the numerical precision of computations (e.g., float32 or float16) to balance speed, memory usage, and accuracy.

In [3]:
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.unk_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cpu",
    torch_dtype=torch.float32
)

Loading model...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Step 4: Initialize conversation messages

In modern chat-based LLMs, we use a structured conversation format made of messages. Each message has a specific role that tells the model who is speaking and how to behave.

messages: This is the full conversation history between the user and the AI. Each message has two parts: role who is speaking and content what they are saying

There are three types of roles in a chat-based AI system. The system role defines the rules and behavior of the AI, such as how it should respond. The user role represents the questions or inputs given by the person using the chatbot. The assistant role contains the AI’s responses generated based on both the system instructions and user input, forming the conversation flow.

In [4]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."
    }
]

Step 5.1: Update conversation history
Store the user message in the conversation history.

In [7]:
print("Chatbot started. Type 'exit' to quit.\n")
while True:
  user_input = input("> ")

  if user_input.lower() == "exit":
      break

messages.append({"role": "user", "content": user_input})
messages = [messages[0]] + messages[-10:]

Chatbot started. Type 'exit' to quit.

> hello
> exit


Step 5.2 : Apply chat template
Modern Hugging Face chat models use chat templates to format conversations automatically.

apply_chat_template(): This function converts the structured message format (system, user, assistant) into a single properly formatted prompt that the model can understand. It ensures the conversation follows the model’s expected template, including special tokens and structure.

tokenize: converts text into tokens

add_generation_prompts: signals the model to generate a reply This flag tells the model that the input ends here and it should now start generating a response. It ensures the model knows where the assistant’s reply should begin in the conversation format.

return_tensors: returns PyTorch tensors

In [9]:
tokenized = tokenizer.apply_chat_template(
  messages,
  tokenize=True,
  add_generation_prompt=True,
  return_tensors="pt",
  return_dict=True,
  max_length=512
  )

Step 5.3: Generate response
This code is used to generate a response from a language model based on the given input text.It takes your tokenized input and makes the model predict and generate a reply token-by-token, while controlling the style, length, and randomness of the output

with torch.inference_mode():Runs the code in inference mode (no training). It makes generation faster and memory-efficient.

model.generate(): This is the function that actually makes the model produce a response based on the input tokens.

tokenized["input_ids"]: These are the numerical tokens representing your input text. The model uses them as the starting point for generating output.

attention_mask=tokenized["attention_mask"] : This tells the model which tokens are real words and which are padding. It ensures the model ignores padding tokens during generation.

pad_token_id=tokenizer.pad_token_id :Specifies the padding token ID so the model knows how to handle padding during generation, preventing errors or warnings.

In [10]:
with torch.inference_mode():
      outputs = model.generate(
          tokenized["input_ids"],
          attention_mask=tokenized["attention_mask"],
          max_new_tokens=60,
          temperature=0.5,
          top_p=0.8,
          do_sample=True,
          repetition_penalty=1.3,
          no_repeat_ngram_size=3,
          pad_token_id=tokenizer.pad_token_id
      )

Step 5.4: Decode and display response
After the model generates output, it is still in token form (numbers). This step converts it back into readable text and shows it to the user.

outputs[0][tokenized["input_ids"].shape[-1]:] This part extracts only the newly generated response from the model.

outputs[0] :full sequence (input + generated text)

tokenized["input_ids"].shape[-1] : length of original input

**[...] **: slices out only the generated part (removes the input)

In [11]:
response = tokenizer.decode(
    outputs[0][tokenized["input_ids"].shape[-1]:],
    skip_special_tokens=True
  )
print(f"Bot: {response}\n")

Bot: To exit, simply type 'exit' or press the Esc key on your keyboard to close all applications simultaneously. Alternatively, you can use Ctrl+Alt+Esc for quick access to this option from other apps as well.



Step 5.5 : Save assistant response

In [12]:
messages.append({"role": "assistant", "content": response})

In [13]:
messages = [{
    "role": "system",
    "content": "You are a very friendly and cheerful assistant. Always respond in a warm, casual, and encouraging tone."
}]

In [14]:
temperature=0.9
top_p=0.95

Full Code

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token

model = AutoModelForCausalLM.from_pretrained(
  model_name,
  device_map="cpu",
  torch_dtype=torch.float32
)

messages = [
  {
      "role": "system",
      "content": "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."
  }
]

print("Chatbot started. Type 'exit' to quit.\n")
while True:
  user_input = input("> ")

  if user_input.lower() == "exit":
      break

  messages.append({"role": "user", "content": user_input})

  messages = [messages[0]] + messages[-10:]

  tokenized = tokenizer.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True,
      max_length=512
  )

  with torch.inference_mode():
      outputs = model.generate(
          tokenized["input_ids"],
          attention_mask=tokenized["attention_mask"],
          max_new_tokens=60,
          temperature=0.5,
          top_p=0.8,
          do_sample=True,
          repetition_penalty=1.3,
          no_repeat_ngram_size=3,
          pad_token_id=tokenizer.pad_token_id
      )

  response = tokenizer.decode(
      outputs[0][tokenized["input_ids"].shape[-1]:],
      skip_special_tokens=True
  )

  print(f"Bot: {response}\n")

  messages.append({"role": "assistant", "content": response})


Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Chatbot started. Type 'exit' to quit.

> hello; what is the breaking news today
Bot: the latest developments on current events, including political updates or significant happenings around you can be found online at your preferred sources such as national newspapers websites like abcnews.com , cbsnews.org . these sites provide real time information about global issues from various perspectives to keep up with important matters that may



KeyboardInterrupt: Interrupted by user